# ST3 — Energy Policy: Data Ingestion & Cleaning
**Team 7 Lambda | Phase 1 | Feb 27 – Mar 2**

**Goal:** Pull, clean, and save two data sources:
1. OWID Energy Data — energy mix by country/year (auto-downloadable)
2. RFF World Carbon Pricing Database — carbon tax / ETS prices by country/year

**Causal role:** ST3 quantifies asymmetric national energy policies that amplify downstream mineral demand distortions (ST2 → ST1 → ST3 → ST4).

**Output:** `data/processed/st3_energy.parquet`, `data/processed/st3_carbon_price.parquet`

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import requests
from pathlib import Path

from config import DATA_RAW, DATA_PROC, FOCAL_COUNTRIES, START_YEAR, END_YEAR
from utils import normalize_countries, save_parquet, set_style, log

set_style()
log.info('ST3 setup complete.')

## 3A — OWID Energy Data

**Source:** https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv  
**Auto-download:** Yes — the cell below fetches it directly from GitHub.  
**Key columns:** `renewables_share_energy`, `fossil_share_energy`, `nuclear_share_energy`, `solar_electricity`, `wind_electricity`

Coverage: ~200 countries, 1965–2023, annual.

In [ ]:
# ── Download OWID Energy Data ───────────────────────────────────────────────
OWID_URL  = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
OWID_DIR  = DATA_RAW / "owid"
OWID_PATH = OWID_DIR / "owid-energy-data.csv"
OWID_DIR.mkdir(parents=True, exist_ok=True)

if not OWID_PATH.exists():
    log.info("Downloading OWID energy data (~10 MB)...")
    try:
        r = requests.get(OWID_URL, timeout=120)
        r.raise_for_status()
        OWID_PATH.write_bytes(r.content)
        log.info(f"Saved OWID CSV: {OWID_PATH}")
    except Exception as e:
        log.error(f"OWID download failed: {e}")
else:
    log.info(f"OWID file exists: {OWID_PATH}")

In [ ]:
# ── Load and Clean OWID Energy Data ────────────────────────────────────────

# Columns we need from the OWID dataset
OWID_COLS = [
    "country", "year",
    "renewables_share_energy",     # % of primary energy from renewables
    "fossil_share_energy",         # % from fossil fuels
    "nuclear_share_energy",        # % from nuclear
    "solar_electricity",           # TWh solar power generation
    "wind_electricity",            # TWh wind power generation
    "hydro_electricity",           # TWh hydro power generation
    "electricity_generation",      # TWh total electricity generated
    "primary_energy_consumption",  # TWh total primary energy
]


def load_owid(path: Path) -> pd.DataFrame:
    """
    Load OWID energy CSV; filter to analysis window and non-aggregate entities.
    Normalizes country names to ISO-3. Drops continental/world aggregates.

    Args:
        path: Path to owid-energy-data.csv

    Returns:
        DataFrame with OWID_COLS plus 'iso3'. Empty DataFrame if file missing.

    Usage:
        owid_df = load_owid(OWID_PATH)
    """
    if not path.exists():
        log.warning(f"OWID file not found at {path}. Run the download cell first.")
        return pd.DataFrame()

    # Only load columns that exist in the file to avoid KeyErrors
    available = pd.read_csv(path, nrows=0).columns.tolist()
    cols_to_load = [c for c in OWID_COLS if c in available]
    missing = set(OWID_COLS) - set(cols_to_load)
    if missing:
        log.warning(f"OWID columns not present in file: {missing}")

    raw = pd.read_csv(path, usecols=cols_to_load, low_memory=False)

    # Filter analysis window
    raw["year"] = pd.to_numeric(raw["year"], errors="coerce")
    raw = raw[(raw["year"] >= START_YEAR) & (raw["year"] <= END_YEAR)].copy()

    # Normalize country → ISO-3 (drops world/continent aggregates that can't map)
    raw = normalize_countries(raw, "country")
    raw = raw.dropna(subset=["iso3"])

    # Coerce numeric columns
    numeric_cols = [c for c in cols_to_load if c not in ("country", "year")]
    for col in numeric_cols:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    log.info(f"OWID loaded: {raw.shape} | unique countries: {raw['iso3'].nunique()}")
    return raw.reset_index(drop=True)


owid_df = load_owid(OWID_PATH)

if not owid_df.empty:
    display(owid_df[["iso3", "country", "year",
                     "renewables_share_energy", "fossil_share_energy",
                     "solar_electricity", "wind_electricity"]].head(10))

In [ ]:
# ── Build st3_energy Schema ─────────────────────────────────────────────────

def build_st3_energy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshape OWID data into the st3_energy target schema.
    Renames columns, forward-fills gaps up to 2 periods (per project rules),
    and returns cleaned DataFrame.

    Target schema:
        iso3, year, renewables_share, fossil_share, nuclear_share,
        solar_twh, wind_twh, hydro_twh, electricity_twh, primary_energy_twh

    Args:
        df: output of load_owid()

    Returns:
        Cleaned DataFrame in target schema. Empty stub if input is empty.

    Usage:
        st3_energy = build_st3_energy(owid_df)
    """
    TARGET_COLS = ["iso3", "year", "renewables_share", "fossil_share",
                   "nuclear_share", "solar_twh", "wind_twh",
                   "hydro_twh", "electricity_twh", "primary_energy_twh"]

    if df.empty:
        return pd.DataFrame(columns=TARGET_COLS)

    rename_map = {
        "renewables_share_energy":    "renewables_share",
        "fossil_share_energy":        "fossil_share",
        "nuclear_share_energy":       "nuclear_share",
        "solar_electricity":          "solar_twh",
        "wind_electricity":           "wind_twh",
        "hydro_electricity":          "hydro_twh",
        "electricity_generation":     "electricity_twh",
        "primary_energy_consumption": "primary_energy_twh",
    }

    # Only rename columns that actually exist
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    out = df[["iso3", "year"] + list(existing.keys())].rename(columns=existing)

    # Forward-fill gaps — max 2 periods (project hard rule)
    out = out.sort_values(["iso3", "year"])
    numeric_cols = [c for c in out.columns if c not in ("iso3", "year")]
    out[numeric_cols] = (
        out.groupby("iso3")[numeric_cols]
        .transform(lambda s: s.ffill(limit=2))
    )

    # Ensure all target cols present (fill missing with NaN)
    for col in TARGET_COLS:
        if col not in out.columns:
            out[col] = np.nan

    log.info(f"st3_energy built: {out.shape}")
    return out[TARGET_COLS].reset_index(drop=True)


st3_energy = build_st3_energy(owid_df)

if not st3_energy.empty:
    log.info(f"Countries with >50% renewables share (latest year): "
             f"{st3_energy[st3_energy['renewables_share'] > 50]['iso3'].nunique()}")
    display(st3_energy.describe().round(2))

## 3B — RFF World Carbon Pricing Database

**Source:** https://github.com/g-dolphin/WorldCarbonPricingDatabase  
**Auto-download:** Yes — fetched directly from GitHub raw.  
**Key columns:** `jurisdiction` (country), `year`, `tax` (carbon tax rate), ETS price  

> **Note on currency:** Carbon tax rates are reported in domestic currency per tCO₂e.  
> This notebook captures relative magnitudes and binary presence (`has_carbon_price`).  
> USD conversion requires historical exchange rates (out of scope for ingestion phase).

Coverage: ~45 jurisdictions, 1990–2022.

In [ ]:
# ── Download RFF Carbon Pricing Database ───────────────────────────────────
# Try multiple URL patterns (repo structure changed over time)
RFF_CANDIDATES = [
    "https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/_dataset/data/WorldCarbonPricingDatabase.csv",
    "https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/_raw/data/WorldCarbonPricingDatabase.csv",
    "https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/master/WorldCarbonPricingDatabase.csv",
]

RFF_DIR  = DATA_RAW / "rff"
RFF_PATH = RFF_DIR / "WorldCarbonPricingDatabase.csv"
RFF_DIR.mkdir(parents=True, exist_ok=True)

if not RFF_PATH.exists():
    downloaded = False
    for url in RFF_CANDIDATES:
        try:
            log.info(f"Trying: {url}")
            r = requests.get(url, timeout=60)
            if r.status_code == 200 and len(r.content) > 1000:
                RFF_PATH.write_bytes(r.content)
                log.info(f"Saved RFF data: {RFF_PATH} ({len(r.content):,} bytes)")
                downloaded = True
                break
            else:
                log.warning(f"URL returned {r.status_code} or empty response")
        except Exception as e:
            log.warning(f"Failed: {e}")
    if not downloaded:
        log.warning(
            "Could not auto-download RFF data. "
            "Manually download from https://github.com/g-dolphin/WorldCarbonPricingDatabase "
            f"and place CSV at {RFF_PATH}. Will create empty stub."
        )
else:
    log.info(f"RFF file exists: {RFF_PATH}")

In [ ]:
# ── Load and Clean RFF Carbon Pricing Data ─────────────────────────────────

def load_rff_carbon(path: Path) -> pd.DataFrame:
    """
    Load RFF World Carbon Pricing Database CSV.
    Aggregates across sectors to get the maximum carbon price per country-year.

    Output columns:
        iso3, year, tax_price_local, ets_price_local, has_carbon_price

    Note: tax_price_local / ets_price_local are in domestic currency per tCO2e.
    has_carbon_price = True if any mechanism > 0 in that country-year.

    Args:
        path: Path to WorldCarbonPricingDatabase.csv

    Returns:
        Cleaned DataFrame. Empty DataFrame if file missing or unreadable.

    Usage:
        carbon_df = load_rff_carbon(RFF_PATH)
    """
    if not path.exists():
        log.warning(f"RFF file not found at {path}.")
        return pd.DataFrame()

    try:
        df = pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    except Exception as e:
        log.error(f"Could not parse RFF CSV: {e}")
        return pd.DataFrame()

    log.info(f"RFF raw: {df.shape} | columns: {list(df.columns[:20])}")

    # Normalize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

    # Country column may be 'jurisdiction' or 'country'
    country_col = next((c for c in df.columns if c in ("jurisdiction", "country")), None)
    if country_col is None:
        log.error("No country/jurisdiction column found in RFF data.")
        return pd.DataFrame()
    df = df.rename(columns={country_col: "country"})

    df = normalize_countries(df, "country")
    df = df.dropna(subset=["iso3"])

    # Year filter
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

    # Identify tax and ETS price columns (naming varies by version)
    tax_col = next(
        (c for c in df.columns if c == "tax" or ("tax" in c and "rate" not in c and "type" not in c)),
        None
    )
    ets_col = next(
        (c for c in df.columns if "ets" in c and "price" in c),
        next((c for c in df.columns if c == "ets"), None)
    )

    log.info(f"Identified tax_col={tax_col}, ets_col={ets_col}")

    if tax_col:
        df[tax_col] = pd.to_numeric(df[tax_col], errors="coerce").fillna(0)
    if ets_col:
        df[ets_col] = pd.to_numeric(df[ets_col], errors="coerce").fillna(0)

    # Aggregate: max price across all sectors per country-year
    agg_dict = {}
    if tax_col:
        agg_dict[tax_col] = "max"
    if ets_col:
        agg_dict[ets_col] = "max"

    if agg_dict:
        out = df.groupby(["iso3", "year"]).agg(agg_dict).reset_index()
        out = out.rename(columns={
            **(  {tax_col: "tax_price_local"} if tax_col else {}),
            **(  {ets_col: "ets_price_local"} if ets_col else {}),
        })
    else:
        # No price columns found — flag presence only
        log.warning("No tax/ETS price columns found. Flagging presence only.")
        out = df.groupby(["iso3", "year"]).size().reset_index(name="_count")
        out = out.drop(columns=["_count"])

    # Ensure both price cols exist
    for col in ("tax_price_local", "ets_price_local"):
        if col not in out.columns:
            out[col] = np.nan

    out["has_carbon_price"] = (
        (out["tax_price_local"].fillna(0) > 0) |
        (out["ets_price_local"].fillna(0) > 0)
    )

    n_priced = out[out["has_carbon_price"]]["iso3"].nunique()
    log.info(f"RFF cleaned: {out.shape} | countries with carbon pricing: {n_priced}")
    return out[["iso3", "year", "tax_price_local", "ets_price_local", "has_carbon_price"]]


carbon_df = load_rff_carbon(RFF_PATH)

if not carbon_df.empty:
    display(carbon_df[carbon_df["has_carbon_price"]].sort_values(["iso3", "year"]).head(15))

In [ ]:
# ── Save All ST3 Outputs to Parquet ────────────────────────────────────────

# st3_energy
if not st3_energy.empty:
    save_parquet(st3_energy, DATA_PROC / "st3_energy.parquet", "ST3 Energy Mix")
else:
    log.warning("st3_energy is empty — check OWID download. Saving empty stub.")
    stub_energy = pd.DataFrame(columns=[
        "iso3", "year", "renewables_share", "fossil_share", "nuclear_share",
        "solar_twh", "wind_twh", "hydro_twh", "electricity_twh", "primary_energy_twh"
    ])
    save_parquet(stub_energy, DATA_PROC / "st3_energy.parquet", "ST3 Energy Mix (stub)")

# st3_carbon_price
if not carbon_df.empty:
    save_parquet(carbon_df, DATA_PROC / "st3_carbon_price.parquet", "ST3 Carbon Pricing")
else:
    log.warning("carbon_df is empty — check RFF download. Saving empty stub.")
    stub_carbon = pd.DataFrame(columns=[
        "iso3", "year", "tax_price_local", "ets_price_local", "has_carbon_price"
    ])
    save_parquet(stub_carbon, DATA_PROC / "st3_carbon_price.parquet", "ST3 Carbon Pricing (stub)")

log.info("Phase 1 ST3 complete. Check data/processed/ for output files.")